# 三模型软投票与 B06 概率校准

比较 B06、TextCNN、DistilBERT 的硬投票、等权软投票和加权软投票。B06 使用训练集五折的折外决策分数拟合 sigmoid 概率校准，每折单独拟合 TF-IDF 和 SVM。推理使用全训练集重新拟合的 TF-IDF 与 SVM，并核对其决策分数与原 B06 一致。TextCNN 与 DistilBERT 沿用原检查点的 softmax 概率。

三模型的等权均为 1/3。加权搜索采用步长 0.1、每个模型权重至少 0.1、总和为 1 的组合，另纳入等权。按验证 F1 选择权重，同分优先接近等权。比较全验证集上选择并报告的结果，以及在验证集上五折选权重后拼接折外预测的结果。正类概率严格大于 0.5 时输出正类。

验证集此前已用于模型选择，折外比较仅隔离本轮权重选择与每折计分。实验仅读取训练集和验证集。

[scikit-learn 概率校准 API](https://scikit-learn.org/1.8/modules/generated/sklearn.calibration.CalibratedClassifierCV.html)


In [ ]:
import hashlib
import json
import math

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.svm import LinearSVC
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.paths import ROOT, CONFIGS_DIR, DATA_DIR, TABLES_DIR
from sentiment_analysis.sequence_data import SentimentDataset, TOKEN_PATTERN
from sentiment_analysis.textcnn import TextCNN

def sha256(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

protocol = json.loads((CONFIGS_DIR / "final_test_protocol.json").read_text(encoding="utf-8"))
for name in ["train", "validation", "tfidf", "textcnn"]:
    record = protocol["files"][name]
    if sha256(ROOT / record["path"]) != record["sha256"]:
        raise ValueError(f"Fixed input changed: {name}")


In [ ]:
# 恢复 B06 并核对验证结果
train_data = pd.read_csv(ROOT / protocol["files"]["train"]["path"])
validation_data = pd.read_csv(ROOT / protocol["files"]["validation"]["path"])
vectorizer = joblib.load(ROOT / protocol["files"]["tfidf"]["path"])
X_train = vectorizer.transform(train_data["text"])
X_validation = vectorizer.transform(validation_data["text"])
svm_model = LinearSVC(C=1.0, max_iter=1000, random_state=42)
svm_model.fit(X_train, train_data["label"])
validation_prediction = svm_model.predict(X_validation)
validation_metrics = classification_metrics(validation_data["label"], validation_prediction)
baseline_record = pd.read_csv(ROOT / protocol["files"]["baseline_validation"]["path"])
baseline_record = baseline_record[baseline_record["EXPERIMENT"] == "B06"].iloc[0]
for metric, value in validation_metrics.items():
    if not np.isclose(value, baseline_record[metric], rtol=0, atol=1e-12):
        raise ValueError(f"B06 validation metric does not match: {metric}")
print("B06 validation results verified.")


In [ ]:
# 每个训练折独立学习词表和模型，使用折外分数校准概率
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(clone(vectorizer), clone(svm_model))
calibration_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
calibrated_b06 = CalibratedClassifierCV(
    estimator=pipeline, method="sigmoid", cv=calibration_cv, ensemble=False,
)
calibrated_b06.fit(train_data["text"], train_data["label"])
refitted = calibrated_b06.calibrated_classifiers_[0].estimator
np.testing.assert_allclose(
    refitted.decision_function(validation_data["text"]),
    svm_model.decision_function(X_validation), rtol=0, atol=1e-7,
)
np.testing.assert_array_equal(refitted.predict(validation_data["text"]), validation_prediction)
positive_index = list(calibrated_b06.classes_).index(1)
svm_probabilities = calibrated_b06.predict_proba(validation_data["text"])[:, positive_index]
labels = validation_data["label"].to_numpy()
print(f"B06 calibrated validation Brier: {brier_score_loss(labels, svm_probabilities):.4f}")
print(f"B06 calibrated validation log loss: {log_loss(labels, svm_probabilities):.4f}")
print("Train-only calibration complete. Refit margins match B06.", flush=True)


In [ ]:
# 恢复 TextCNN 的原始预测和正类概率
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = torch.load(ROOT / protocol["files"]["textcnn"]["path"], map_location="cpu", weights_only=True)
if cnn["token_pattern"] != TOKEN_PATTERN.pattern or not cnn["lowercase"]:
    raise ValueError("TextCNN tokenizer does not match the evaluation.")
cnn_model = TextCNN(**cnn["model_config"]).to(device)
cnn_model.load_state_dict(cnn["model_state_dict"])
cnn_model.eval()
dataset = SentimentDataset(validation_data, cnn["vocabulary"], cnn["max_length"])
loader = DataLoader(dataset, batch_size=32, shuffle=False)
cnn_probabilities = []
observed_labels = []
with torch.inference_mode():
    for token_ids, lengths, batch_labels in loader:
        logits = cnn_model(token_ids.to(device), lengths.to(device))
        cnn_probabilities.extend(logits.softmax(dim=1)[:, 1].cpu().tolist())
        observed_labels.extend(batch_labels.tolist())
if not np.array_equal(observed_labels, labels):
    raise ValueError("TextCNN validation rows are misaligned.")


In [ ]:
# 使用 DistilBERT 固定检查点和原评估配置
model_dir = DATA_DIR / "models" / "distilbert_seed42"
protocol = json.loads((CONFIGS_DIR / "distilbert_test_protocol.json").read_text(encoding="utf-8"))
training_path = CONFIGS_DIR / "distilbert_finetuning.json"
training = json.loads(training_path.read_text(encoding="utf-8"))
if (sha256(training_path) != protocol["training_protocol_sha256"]
        or sha256(ROOT / "data/cleaned/val.csv") != training["split_sha256"]["validation"]):
    raise ValueError("DistilBERT validation data or training protocol changed.")
for name, digest in protocol["checkpoint_sha256"].items():
    if sha256(model_dir / name) != digest:
        raise ValueError(f"DistilBERT checkpoint changed: {name}")
if device.type != "cuda" or not torch.cuda.is_bf16_supported():
    raise RuntimeError("The fixed DistilBERT evaluation requires CUDA BF16.")
# 先恢复已选检查点并核对验证指标
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, trust_remote_code=False)
model = AutoModelForSequenceClassification.from_pretrained(
    model_dir, local_files_only=True, use_safetensors=True, trust_remote_code=False,
    attn_implementation=protocol["attention_implementation"],
).to(device)
model.eval()
if model.config.id2label != {0: "NEGATIVE", 1: "POSITIVE"}:
    raise ValueError("Checkpoint label mapping does not match the evaluation.")

def predict(data, name):
    labels = []
    predictions = []
    probabilities = []
    loss_sum = 0.0
    truncated = 0
    batch_size = protocol["batch_size"]
    with torch.inference_mode():
        for start in range(0, len(data), batch_size):
            rows = data.iloc[start:start + batch_size]
            texts = rows["text"].tolist()
            lengths = tokenizer(texts, truncation=False, return_length=True, verbose=False)["length"]
            truncated += sum(length > protocol["max_length"] for length in lengths)
            batch = tokenizer(
                texts, truncation=True, max_length=protocol["max_length"], padding=True,
                pad_to_multiple_of=protocol["pad_to_multiple_of"], return_tensors="pt",
                return_token_type_ids=False,
            ).to(device)
            batch_labels = torch.tensor(rows["label"].tolist(), dtype=torch.long, device=device)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                output = model(**batch, labels=batch_labels)
            loss_sum += float(output.loss) * len(rows)
            labels.extend(batch_labels.cpu().tolist())
            predictions.extend(output.logits.argmax(dim=1).cpu().tolist())
            probabilities.extend(output.logits.float().softmax(dim=1)[:, 1].cpu().tolist())
            if start and start % (batch_size * 500) == 0:
                print(f"{name}: {start:,}/{len(data):,}", flush=True)
    if labels != data["label"].tolist() or not math.isfinite(loss_sum):
        raise ValueError(f"{name} labels or loss are invalid.")
    return loss_sum / len(labels), predictions, probabilities, truncated



_, distilbert_predictions, distilbert_probabilities, _ = predict(validation_data, "Validation")


In [ ]:
# 复现原硬投票，在相同样本上建立概率矩阵
hard_vote = (
    validation_prediction
    + (np.array(cnn_probabilities) > 0.5).astype(int)
    + np.array(distilbert_predictions)
    >= 2
).astype(int)
reference = pd.read_csv(TABLES_DIR / "three_model_voting_validation.csv").set_index("MODEL")
previous_vote = reference.loc["B06 + TextCNN + DistilBERT majority"]
for metric, value in classification_metrics(labels, hard_vote).items():
    if not math.isclose(value, previous_vote[metric], abs_tol=1e-12):
        raise ValueError(f"Original hard vote did not reproduce: {metric}")

probabilities = np.column_stack([svm_probabilities, cnn_probabilities, distilbert_probabilities])
if probabilities.shape != (len(labels), 3) or not np.isfinite(probabilities).all():
    raise ValueError("Invalid probability matrix.")
if (probabilities < 0).any() or (probabilities > 1).any():
    raise ValueError("Probabilities must be between zero and one.")

equal_weights = np.full(3, 1 / 3)
grid = [equal_weights] + [
    np.array([svm, cnn, 10 - svm - cnn]) / 10
    for svm in range(1, 9) for cnn in range(1, 10 - svm)
]
distances = [float(np.square(weights - equal_weights).sum()) for weights in grid]

def soft_vote(scores, weights):
    return (scores @ weights > 0.5).astype(int)

def select_weights(scores, targets):
    f1_values = [f1_score(targets, soft_vote(scores, weights)) for weights in grid]
    best_index = max(range(len(grid)), key=lambda index: (f1_values[index], -distances[index], -index))
    return grid[best_index]

selected_weights = select_weights(probabilities, labels)
cross_fitted_predictions = np.full(len(labels), -1, dtype=int)
fold_rows = []
weight_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (fit_rows, eval_rows) in enumerate(weight_cv.split(probabilities, labels), start=1):
    weights = select_weights(probabilities[fit_rows], labels[fit_rows])
    cross_fitted_predictions[eval_rows] = soft_vote(probabilities[eval_rows], weights)
    fold_rows.append({
        "FOLD": fold, "EVAL ROWS": len(eval_rows), "B06 WEIGHT": weights[0],
        "TEXTCNN WEIGHT": weights[1], "DISTILBERT WEIGHT": weights[2],
    })
if (cross_fitted_predictions < 0).any():
    raise ValueError("Some validation rows did not receive a fold prediction.")


In [ ]:
# 保存指标、选定权重与纠错数
rows = []
hard_correct = hard_vote == labels
for name, prediction, weights in [
    ("DistilBERT", np.array(distilbert_predictions), None),
    ("Hard vote", hard_vote, None),
    ("Equal soft vote", soft_vote(probabilities, equal_weights), equal_weights),
    ("Weighted soft vote (same validation)", soft_vote(probabilities, selected_weights), selected_weights),
    ("Weighted soft vote (out of fold)", cross_fitted_predictions, None),
]:
    correct = prediction == labels
    rows.append({
        "MODEL": name, **classification_metrics(labels, prediction),
        "B06 WEIGHT": None if weights is None else weights[0],
        "TEXTCNN WEIGHT": None if weights is None else weights[1],
        "DISTILBERT WEIGHT": None if weights is None else weights[2],
        "CORRECTED VS HARD": int((~hard_correct & correct).sum()),
        "NEW ERRORS VS HARD": int((hard_correct & ~correct).sum()),
    })

comparison = pd.DataFrame(rows)
fold_weights = pd.DataFrame(fold_rows)
comparison.to_csv(TABLES_DIR / "three_model_soft_voting_validation.csv", index=False)
fold_weights.to_csv(TABLES_DIR / "three_model_soft_voting_fold_weights.csv", index=False)
model_path = DATA_DIR / "models" / "b06_sigmoid_cv5.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(calibrated_b06, model_path)
display(comparison.round(4))
display(fold_weights.round(4))


In [ ]:
# 保存逐条验证集预测，供后续消融实验复用
prediction_table = pd.DataFrame({
    "VALIDATION ROW": np.arange(len(labels)),
    "LABEL": labels,
    "B06 POSITIVE PROBABILITY": svm_probabilities,
    "TEXTCNN POSITIVE PROBABILITY": cnn_probabilities,
    "DISTILBERT POSITIVE PROBABILITY": distilbert_probabilities,
    "Hard vote": hard_vote,
})
prediction_table.to_csv(TABLES_DIR / "three_model_soft_voting_validation_predictions.csv", index=False)

## 验证结果

B06 校准后的 Brier 分数为 0.0810，Log Loss 为 0.2738。校准流程重新拟合的 SVM 与原 B06 在验证集上的类别预测逐条一致。

| 方案 | 准确率 | F1 | 纠正硬投票错误 | 新增错误 |
| --- | ---: | ---: | ---: | ---: |
| 三模型硬投票 | 91.73% | 91.79% | 0 | 0 |
| 等权软投票 | 92.45% | 92.52% | 34 | 16 |
| 加权软投票（同集选权和评分） | 92.81% | 92.87% | 48 | 21 |
| 加权软投票（折外评估） | 92.65% | 92.71% | 46 | 23 |

按 B06、TextCNN、DistilBERT 的顺序，全验证集选出权重 (0.2, 0.3, 0.5)。五次折外评估中，四次选出这组权重，一次选出 (0.3, 0.2, 0.5)。折外加权比硬投票高 0.91 个 F1 百分点，比等权高 0.19 个百分点。折外评估衡量的是在每折重新选权的流程，固定权重 (0.2, 0.3, 0.5) 的表现仍需独立数据验证。
